# Perseptron v2 - 03 Multimodal Late Fusion

Bu notebook proposal'in ana modelini egitir. Tabular branch musteri/urun metadata'sini, visual branch ise aday urun embedding'i ve musteri gorsel gecmis profilini isler. Iki temsil late fusion ile birlestirilir.

Ana hipotez bu model ile test edilecek: late fusion, tek modaliteli baseline'lari MAP@12 tarafinda gecmeli.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

def find_source_project():
    candidates = [Path('/kaggle/working/perseptron_project'), Path.cwd(), Path('/kaggle/input/perseptron-project')]
    candidates += [path for path in Path('/kaggle/input').glob('*') if path.is_dir()]
    for path in candidates:
        if (path / 'src' / 'proposal_v2').exists():
            return path
    return Path.cwd()

SOURCE_PROJECT_DIR = find_source_project()
WORK_PROJECT_DIR = Path('/kaggle/working/perseptron_project_work') if Path('/kaggle').exists() else SOURCE_PROJECT_DIR
if SOURCE_PROJECT_DIR != WORK_PROJECT_DIR:
    shutil.copytree(SOURCE_PROJECT_DIR, WORK_PROJECT_DIR, dirs_exist_ok=True)

PROJECT_DIR = WORK_PROJECT_DIR
os.environ['PERSEPTRON_PROJECT_DIR'] = str(PROJECT_DIR)
os.environ['PYTHONPATH'] = str(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('SOURCE_PROJECT_DIR =', SOURCE_PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)

def restore_previous_outputs():
    if not Path('/kaggle/input').exists():
        return
    for input_root in Path('/kaggle/input').glob('*'):
        if input_root == SOURCE_PROJECT_DIR:
            continue
        for relative in ['reports/proposal_v2', 'models/proposal_v2']:
            source = input_root / relative
            target = PROJECT_DIR / relative
            if source.exists():
                target.mkdir(parents=True, exist_ok=True)
                shutil.copytree(source, target, dirs_exist_ok=True)
                print('Restored', source, '->', target)

restore_previous_outputs()

def run_module(module, *args):
    command = [sys.executable, '-m', module, *map(str, args)]
    print('RUN:', ' '.join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)


## Parametreler

Final rapor icin fold 0-4 ayri ayri calistirilacak. Her fold ayni split dosyasini kullanir.

In [ ]:
FAST_RUN = True
FOLD_ID = 0
MAX_TRAIN_POSITIVES = 5_000 if FAST_RUN else 200_000
MAX_VAL_POSITIVES = 1_000 if FAST_RUN else 50_000
EPOCHS = 1 if FAST_RUN else 3
BATCH_SIZE = 512 if FAST_RUN else 4096


## Egitim

Bu hucre sadece `late_fusion` modelini egitir. Sonradan ranking notebookunda ayni candidate havuzu uzerinde tabular-only ve image-history ile karsilastirilir.

In [ ]:
run_module(
    'src.proposal_v2.train',
    '--fold-id', FOLD_ID,
    '--models', 'late_fusion',
    '--max-train-positives', MAX_TRAIN_POSITIVES,
    '--max-val-positives', MAX_VAL_POSITIVES,
    '--epochs', EPOCHS,
    '--batch-size', BATCH_SIZE,
)
